# VyapaarIQ retail demo data

Run the code cell below to populate `vyapaariq.db` with about 6.5 months of realistic retail-store demo data while keeping `users`, `otp_verification`, and `business_profiles` intact.

In [ ]:
import random
import sqlite3
from datetime import date, timedelta

import pandas as pd

random.seed(42)

DB_PATH = 'vyapaariq.db'
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()
cur.execute('PRAGMA foreign_keys = ON')

for table in [
    'stock_alerts',
    'sale_items',
    'sales',
    'purchase_items',
    'purchases',
    'products',
    'customers',
    'suppliers',
    'categories',
    'expenses',
]:
    cur.execute(f'DELETE FROM {table}')

cur.execute("DELETE FROM sqlite_sequence WHERE name IN ('categories','suppliers','customers','products','purchases','purchase_items','sales','sale_items','expenses','stock_alerts')")

profile = cur.execute(
    "SELECT business_name, business_type, city FROM business_profiles ORDER BY id DESC LIMIT 1"
).fetchone()

store_name = profile['business_name'] if profile and profile['business_name'] else 'CityMart Retail Store'
store_type = profile['business_type'] if profile and profile['business_type'] else 'Retail Store'
store_city = profile['city'] if profile and profile['city'] else 'Hyderabad'

categories = [
    ('Groceries', 'Daily household essentials'),
    ('Beverages', 'Soft drinks, juices, and water'),
    ('Snacks', 'Chips, biscuits, and namkeen'),
    ('Dairy', 'Milk, curd, paneer, and butter'),
    ('Personal Care', 'Soap, shampoo, and hygiene products'),
    ('Home Care', 'Cleaning and utility supplies'),
    ('Stationery', 'Basic office and school supplies'),
    ('Packaged Foods', 'Ready to cook and pantry staples'),
]
cur.executemany('INSERT INTO categories(name, description) VALUES (?, ?)', categories)
category_ids = {row['name']: row['id'] for row in cur.execute('SELECT id, name FROM categories').fetchall()}

suppliers = [
    ('ABC Traders', 'Arvind Mehta', '9876500101', 'abc@traders.com', 'Hyderabad', 5),
    ('FreshFarm Distributors', 'Nisha Rao', '9876500102', 'freshfarm@demo.com', 'Hyderabad', 4),
    ('Metro Wholesale Hub', 'Karan Shah', '9876500103', 'metrohub@demo.com', 'Secunderabad', 4),
    ('Prime Home Needs', 'Sonal Jain', '9876500104', 'primehome@demo.com', 'Hyderabad', 5),
    ('QuickSupply Co', 'Rahul Verma', '9876500105', 'quicksupply@demo.com', 'Hyderabad', 4),
]
cur.executemany(
    'INSERT INTO suppliers(name, contact_person, phone, email, city, rating) VALUES (?, ?, ?, ?, ?, ?)',
    suppliers,
)
supplier_ids = {row['name']: row['id'] for row in cur.execute('SELECT id, name FROM suppliers').fetchall()}

customers = []
first_names = ['Amit', 'Priya', 'Rahul', 'Sneha', 'Vikram', 'Neha', 'Rohan', 'Kavya', 'Arjun', 'Pooja', 'Kiran', 'Anita']
last_names = ['Sharma', 'Reddy', 'Verma', 'Patel', 'Gupta', 'Yadav', 'Iyer', 'Mishra', 'Kapoor', 'Jain']
customer_types = ['retail', 'retail', 'retail', 'wholesale']
used_names = set()
for idx in range(40):
    while True:
        name = f"{random.choice(first_names)} {random.choice(last_names)}"
        if name not in used_names:
            used_names.add(name)
            break
    customers.append((
        name,
        f'98{random.randint(10000000, 99999999)}',
        f"customer{idx + 1}@demo.com",
        random.choice([store_city, 'Hyderabad', 'Secunderabad', 'Kukatpally', 'Miyapur']),
        random.choice(customer_types),
    ))

cur.executemany(
    'INSERT INTO customers(name, phone, email, city, customer_type) VALUES (?, ?, ?, ?, ?)',
    customers,
)
customer_ids = [row['id'] for row in cur.execute('SELECT id FROM customers').fetchall()]

product_specs = [
    ('Basmati Rice 5kg', 'Groceries', 'ABC Traders', 'kg', 360, 435, 28, 35),
    ('Wheat Flour 10kg', 'Groceries', 'ABC Traders', 'bag', 320, 395, 24, 30),
    ('Toor Dal 1kg', 'Packaged Foods', 'Metro Wholesale Hub', 'pack', 102, 129, 40, 45),
    ('Sunflower Oil 1L', 'Packaged Foods', 'Metro Wholesale Hub', 'bottle', 118, 149, 38, 45),
    ('Sugar 1kg', 'Groceries', 'ABC Traders', 'pack', 39, 49, 50, 60),
    ('Tea Powder 500g', 'Beverages', 'QuickSupply Co', 'pack', 122, 165, 26, 30),
    ('Instant Coffee 100g', 'Beverages', 'QuickSupply Co', 'jar', 145, 199, 20, 25),
    ('Potato Chips', 'Snacks', 'QuickSupply Co', 'pack', 12, 20, 90, 120),
    ('Salted Biscuits', 'Snacks', 'QuickSupply Co', 'pack', 18, 30, 70, 90),
    ('Chocolate Cookies', 'Snacks', 'QuickSupply Co', 'pack', 22, 35, 60, 80),
    ('Milk 1L', 'Dairy', 'FreshFarm Distributors', 'packet', 46, 58, 55, 70),
    ('Curd 400g', 'Dairy', 'FreshFarm Distributors', 'cup', 28, 40, 35, 45),
    ('Butter 500g', 'Dairy', 'FreshFarm Distributors', 'pack', 210, 255, 18, 20),
    ('Bath Soap 4pc', 'Personal Care', 'Prime Home Needs', 'box', 88, 120, 34, 40),
    ('Shampoo 340ml', 'Personal Care', 'Prime Home Needs', 'bottle', 145, 189, 22, 25),
    ('Toothpaste 150g', 'Personal Care', 'Prime Home Needs', 'tube', 68, 95, 40, 45),
    ('Floor Cleaner 1L', 'Home Care', 'Prime Home Needs', 'bottle', 92, 135, 24, 30),
    ('Dishwash Gel 500ml', 'Home Care', 'Prime Home Needs', 'bottle', 58, 85, 30, 35),
    ('A4 Notebook', 'Stationery', 'Metro Wholesale Hub', 'piece', 26, 40, 45, 50),
    ('Ball Pen Pack', 'Stationery', 'Metro Wholesale Hub', 'pack', 32, 55, 42, 50),
]

products = []
for idx, spec in enumerate(product_specs, start=1):
    name, category, supplier, unit, cost_price, selling_price, opening_stock, reorder_level = spec
    products.append((
        name,
        category_ids[category],
        supplier_ids[supplier],
        f'SKU-{idx:03d}',
        unit,
        cost_price,
        selling_price,
        opening_stock,
        reorder_level,
    ))

cur.executemany(
    '''
    INSERT INTO products(
        name, category_id, supplier_id, sku, unit,
        cost_price, selling_price, current_stock, reorder_level
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''',
    products,
)

product_rows = cur.execute('SELECT * FROM products').fetchall()
products_by_id = {row['id']: dict(row) for row in product_rows}
stock_levels = {pid: row['current_stock'] for pid, row in products_by_id.items()}
product_ids = list(products_by_id.keys())

start_date = date(2025, 10, 1)
end_date = date(2026, 4, 15)
all_dates = [start_date + timedelta(days=offset) for offset in range((end_date - start_date).days + 1)]

def insert_purchase(purchase_date, supplier_id, lines, status='Delivered'):
    total_amount = sum(line['quantity'] * line['unit_cost'] for line in lines)
    cur.execute(
        'INSERT INTO purchases(supplier_id, purchase_date, total_amount, status) VALUES (?, ?, ?, ?)',
        (supplier_id, purchase_date.isoformat(), round(total_amount, 2), status),
    )
    purchase_id = cur.lastrowid
    for line in lines:
        cur.execute(
            'INSERT INTO purchase_items(purchase_id, product_id, quantity, unit_cost) VALUES (?, ?, ?, ?)',
            (purchase_id, line['product_id'], line['quantity'], line['unit_cost']),
        )
        stock_levels[line['product_id']] += line['quantity']

def insert_sale(sale_date, customer_id, lines, payment_method, notes=None):
    total_amount = sum(line['quantity'] * line['price'] - line['discount'] for line in lines)
    cur.execute(
        'INSERT INTO sales(customer_id, sale_date, total_amount, payment_method, notes) VALUES (?, ?, ?, ?, ?)',
        (customer_id, sale_date.isoformat(), round(total_amount, 2), payment_method, notes),
    )
    sale_id = cur.lastrowid
    for line in lines:
        subtotal = round(line['quantity'] * line['price'] - line['discount'], 2)
        cur.execute(
            'INSERT INTO sale_items(sale_id, product_id, quantity, price, discount, subtotal) VALUES (?, ?, ?, ?, ?, ?)',
            (sale_id, line['product_id'], line['quantity'], line['price'], line['discount'], subtotal),
        )
        stock_levels[line['product_id']] -= line['quantity']

supplier_product_map = {}
for pid, row in products_by_id.items():
    supplier_product_map.setdefault(row['supplier_id'], []).append(pid)

opening_purchase_date = start_date - timedelta(days=6)
for supplier_id, supplier_products in supplier_product_map.items():
    lines = []
    for pid in supplier_products:
        lines.append({
            'product_id': pid,
            'quantity': random.randint(90, 150),
            'unit_cost': products_by_id[pid]['cost_price'],
        })
    insert_purchase(opening_purchase_date, supplier_id, lines)

current_month = date(start_date.year, start_date.month, 1)
while current_month <= end_date:
    for restock_day in (3, 17):
        restock_date = current_month.replace(day=restock_day)
        if restock_date < start_date or restock_date > end_date:
            continue
        for supplier_id, supplier_products in supplier_product_map.items():
            lines = []
            for pid in supplier_products:
                if random.random() < 0.75:
                    lines.append({
                        'product_id': pid,
                        'quantity': random.randint(18, 55),
                        'unit_cost': products_by_id[pid]['cost_price'],
                    })
            if lines:
                insert_purchase(restock_date, supplier_id, lines)
    current_month = (current_month.replace(day=28) + timedelta(days=4)).replace(day=1)

for current_date in all_dates:
    if current_date.weekday() in (1, 4):
        supplier_lines = {}
        for pid, row in products_by_id.items():
            if stock_levels[pid] <= row['reorder_level'] + random.randint(10, 25):
                supplier_lines.setdefault(row['supplier_id'], []).append({
                    'product_id': pid,
                    'quantity': random.randint(row['reorder_level'] * 2, row['reorder_level'] * 4),
                    'unit_cost': row['cost_price'],
                })
        for supplier_id, lines in supplier_lines.items():
            if lines:
                insert_purchase(current_date, supplier_id, lines)

payment_methods = ['Cash', 'UPI', 'Card']
sale_notes = ['Walk-in purchase', 'Regular customer order', 'Weekend billing', 'Festival offer applied', 'Neighbourhood delivery']

for current_date in all_dates:
    sale_count = random.randint(12, 20) if current_date.weekday() >= 5 else random.randint(7, 14)
    for _ in range(sale_count):
        chosen_products = random.sample(product_ids, random.randint(1, 4))
        lines = []
        for pid in chosen_products:
            available = max(stock_levels[pid], 0)
            if available <= 0:
                continue
            qty = min(random.randint(1, 4), available)
            if qty <= 0:
                continue
            price = products_by_id[pid]['selling_price']
            discount = 0
            if qty >= 3 and random.random() < 0.35:
                discount = round(price * qty * random.uniform(0.03, 0.08), 2)
            lines.append({
                'product_id': pid,
                'quantity': qty,
                'price': price,
                'discount': discount,
            })
        if lines:
            insert_sale(
                current_date,
                random.choice(customer_ids),
                lines,
                random.choices(payment_methods, weights=[2, 5, 2], k=1)[0],
                random.choice(sale_notes),
            )

expense_rows = []
cursor_date = date(start_date.year, start_date.month, 1)
while cursor_date <= end_date:
    expense_rows.extend([
        ('Rent', random.randint(32000, 38000), cursor_date.replace(day=5), f'{store_name} monthly shop rent'),
        ('Utilities', random.randint(6500, 9200), cursor_date.replace(day=8), 'Electricity and water charges'),
        ('Salaries', random.randint(42000, 56000), cursor_date.replace(day=1), 'Staff salaries for billing and floor support'),
        ('Marketing', random.randint(2500, 7000), cursor_date.replace(day=18), 'Local promotions and discount banners'),
        ('Transport', random.randint(1800, 4200), cursor_date.replace(day=12), 'Goods transport and local delivery support'),
    ])
    cursor_date = (cursor_date.replace(day=28) + timedelta(days=4)).replace(day=1)

expense_rows.extend([
    ('Maintenance', 6400, start_date + timedelta(days=44), 'Refrigerator and shelf maintenance'),
    ('Supplies', 3200, start_date + timedelta(days=97), 'Billing rolls and store packaging materials'),
    ('Repairs', 8700, start_date + timedelta(days=163), 'Minor electrical repair and lighting replacement'),
])

cur.executemany(
    'INSERT INTO expenses(category, amount, expense_date, description) VALUES (?, ?, ?, ?)',
    [(cat, amt, dt.isoformat(), desc) for cat, amt, dt, desc in expense_rows],
)

for pid, qty in stock_levels.items():
    cur.execute('UPDATE products SET current_stock=? WHERE id=?', (qty, pid))

for pid, row in products_by_id.items():
    if stock_levels[pid] <= row['reorder_level']:
        cur.execute(
            'INSERT INTO stock_alerts(product_id, alert_type, threshold, is_active) VALUES (?, ?, ?, ?)',
            (pid, 'Low Stock', row['reorder_level'], 1),
        )

conn.commit()

summary = pd.DataFrame({
    'table': ['categories', 'suppliers', 'customers', 'products', 'purchases', 'purchase_items', 'sales', 'sale_items', 'expenses', 'stock_alerts'],
    'rows': [
        cur.execute('SELECT COUNT(*) FROM categories').fetchone()[0],
        cur.execute('SELECT COUNT(*) FROM suppliers').fetchone()[0],
        cur.execute('SELECT COUNT(*) FROM customers').fetchone()[0],
        cur.execute('SELECT COUNT(*) FROM products').fetchone()[0],
        cur.execute('SELECT COUNT(*) FROM purchases').fetchone()[0],
        cur.execute('SELECT COUNT(*) FROM purchase_items').fetchone()[0],
        cur.execute('SELECT COUNT(*) FROM sales').fetchone()[0],
        cur.execute('SELECT COUNT(*) FROM sale_items').fetchone()[0],
        cur.execute('SELECT COUNT(*) FROM expenses').fetchone()[0],
        cur.execute('SELECT COUNT(*) FROM stock_alerts').fetchone()[0],
    ],
})

sales_range = cur.execute('SELECT MIN(sale_date), MAX(sale_date) FROM sales').fetchone()
purchase_range = cur.execute('SELECT MIN(purchase_date), MAX(purchase_date) FROM purchases').fetchone()

print(f'Seeded demo data for {store_name} ({store_type}) in {store_city}.')
print(f'Sales period: {sales_range[0]} to {sales_range[1]}')
print(f'Purchase period: {purchase_range[0]} to {purchase_range[1]}')
display(summary)

monthly_sales = pd.read_sql_query(
    '''
    SELECT substr(sale_date, 1, 7) AS month,
           COUNT(*) AS invoices,
           ROUND(SUM(total_amount), 2) AS revenue
    FROM sales
    GROUP BY substr(sale_date, 1, 7)
    ORDER BY month
    ''',
    conn,
)
display(monthly_sales)
